# U2Seg Swin-Tiny (Cluster 800) - Colab Ready\n\nNotebook ini menyiapkan pipeline dari nol: clone repo, install dependency, setup data full local, generate/fix annotation, lalu training panjang (resume-able).

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
%cd /content\n!rm -rf U2Seg\n!git clone https://github.com/u2seg/U2Seg.git\n%cd /content/U2Seg

In [ ]:
!python -m pip install -q --upgrade pip setuptools wheel\n!python -m pip install -q fvcore pycocotools scikit-image pillow gdown\n!python -m pip install -q "git+https://github.com/cocodataset/panopticapi.git"\n!python -m pip install -q -e .

In [ ]:
import os\nimport json\nfrom pathlib import Path\n\nU2SEG_ROOT = Path('/content/U2Seg')\nLOCAL_DATA_ROOT = U2SEG_ROOT / 'datasets_local'\nDRIVE_COCO = Path('/content/drive/MyDrive/Datasets/coco')\nDRIVE_U2SEG_ANN = Path('/content/drive/MyDrive/Datasets/U2Seg_Annot/u2seg_annotations')\nDRIVE_PANOPTIC = Path('/content/drive/MyDrive/Datasets/panoptic_anns')\n\nLOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)\nprint('LOCAL_DATA_ROOT =', LOCAL_DATA_ROOT)\nprint('DRIVE_COCO exists =', DRIVE_COCO.exists())\nprint('DRIVE_U2SEG_ANN exists =', DRIVE_U2SEG_ANN.exists())\nprint('DRIVE_PANOPTIC exists =', DRIVE_PANOPTIC.exists())

In [ ]:
# Copy full dataset ke local agar tidak kena timeout Drive.\n!mkdir -p /content/U2Seg/datasets_local/coco\n!mkdir -p /content/U2Seg/datasets_local/u2seg_annotations\n!mkdir -p /content/U2Seg/datasets_local/panoptic_anns\n!rsync -a --delete /content/drive/MyDrive/Datasets/coco/ /content/U2Seg/datasets_local/coco/\n!rsync -a --delete /content/drive/MyDrive/Datasets/U2Seg_Annot/u2seg_annotations/ /content/U2Seg/datasets_local/u2seg_annotations/\n!if [ -d /content/drive/MyDrive/Datasets/panoptic_anns ]; then rsync -a --delete /content/drive/MyDrive/Datasets/panoptic_anns/ /content/U2Seg/datasets_local/panoptic_anns/; fi

In [ ]:
# Wire symlink dataset ke layout yang diharapkan U2Seg/detectron2.\n%cd /content/U2Seg\n!mkdir -p datasets datasets/datasets datasets/prepare_ours\n!rm -rf datasets/coco && ln -s /content/U2Seg/datasets_local/coco datasets/coco\n!rm -rf datasets/prepare_ours/u2seg_annotations && ln -s /content/U2Seg/datasets_local/u2seg_annotations datasets/prepare_ours/u2seg_annotations\n!rm -rf datasets/datasets/coco && ln -s ../coco datasets/datasets/coco\n!rm -rf datasets/datasets/panoptic_anns && ln -s /content/U2Seg/datasets_local/panoptic_anns datasets/datasets/panoptic_anns\n!ls -ld datasets/coco datasets/prepare_ours/u2seg_annotations datasets/datasets/coco datasets/datasets/panoptic_anns

In [ ]:
# Inject Swin helper + train entry + config (non-invasif ke core training script).\nfrom pathlib import Path\n\nPath('/content/U2Seg/tools/swin_helpers.py').write_text('''from detectron2.config import CfgNode as CN\nfrom detectron2.layers import ShapeSpec\nfrom detectron2.modeling.backbone.build import BACKBONE_REGISTRY\nfrom detectron2.modeling.backbone.fpn import FPN, LastLevelMaxPool\nfrom detectron2.modeling.backbone.swin import SwinTransformer\n\n\ndef add_swin_config(cfg):\n    cfg.MODEL.SWIN = CN()\n    cfg.MODEL.SWIN.PRETRAIN_IMG_SIZE = 224\n    cfg.MODEL.SWIN.PATCH_SIZE = 4\n    cfg.MODEL.SWIN.EMBED_DIM = 96\n    cfg.MODEL.SWIN.DEPTHS = [2, 2, 6, 2]\n    cfg.MODEL.SWIN.NUM_HEADS = [3, 6, 12, 24]\n    cfg.MODEL.SWIN.WINDOW_SIZE = 7\n    cfg.MODEL.SWIN.MLP_RATIO = 4.0\n    cfg.MODEL.SWIN.QKV_BIAS = True\n    cfg.MODEL.SWIN.DROP_RATE = 0.0\n    cfg.MODEL.SWIN.ATTN_DROP_RATE = 0.0\n    cfg.MODEL.SWIN.DROP_PATH_RATE = 0.2\n    cfg.MODEL.SWIN.APE = False\n    cfg.MODEL.SWIN.PATCH_NORM = True\n    cfg.MODEL.SWIN.OUT_INDICES = [0, 1, 2, 3]\n    cfg.MODEL.SWIN.USE_CHECKPOINT = False\n\n\n@BACKBONE_REGISTRY.register()\ndef build_swin_fpn_backbone(cfg, input_shape: ShapeSpec):\n    bottom_up = SwinTransformer(\n        pretrain_img_size=cfg.MODEL.SWIN.PRETRAIN_IMG_SIZE,\n        patch_size=cfg.MODEL.SWIN.PATCH_SIZE,\n        in_chans=input_shape.channels,\n        embed_dim=cfg.MODEL.SWIN.EMBED_DIM,\n        depths=cfg.MODEL.SWIN.DEPTHS,\n        num_heads=cfg.MODEL.SWIN.NUM_HEADS,\n        window_size=cfg.MODEL.SWIN.WINDOW_SIZE,\n        mlp_ratio=cfg.MODEL.SWIN.MLP_RATIO,\n        qkv_bias=cfg.MODEL.SWIN.QKV_BIAS,\n        drop_rate=cfg.MODEL.SWIN.DROP_RATE,\n        attn_drop_rate=cfg.MODEL.SWIN.ATTN_DROP_RATE,\n        drop_path_rate=cfg.MODEL.SWIN.DROP_PATH_RATE,\n        ape=cfg.MODEL.SWIN.APE,\n        patch_norm=cfg.MODEL.SWIN.PATCH_NORM,\n        out_indices=cfg.MODEL.SWIN.OUT_INDICES,\n        frozen_stages=cfg.MODEL.BACKBONE.FREEZE_AT,\n        use_checkpoint=cfg.MODEL.SWIN.USE_CHECKPOINT,\n    )\n    return FPN(\n        bottom_up=bottom_up,\n        in_features=cfg.MODEL.FPN.IN_FEATURES,\n        out_channels=cfg.MODEL.FPN.OUT_CHANNELS,\n        norm=cfg.MODEL.FPN.NORM,\n        top_block=LastLevelMaxPool(),\n        fuse_type=cfg.MODEL.FPN.FUSE_TYPE,\n    )\n''')\n\nPath('/content/U2Seg/tools/train_net_swin.py').write_text('''#!/usr/bin/env python\nimport os\nimport sys\nfrom detectron2.config import get_cfg\n\nTOOLS_DIR = os.path.dirname(os.path.abspath(__file__))\nif TOOLS_DIR not in sys.path:\n    sys.path.insert(0, TOOLS_DIR)\n\nimport train_net as base_train\nfrom swin_helpers import add_swin_config\n\n\ndef setup(args):\n    cfg = get_cfg()\n    add_swin_config(cfg)\n    cfg.merge_from_file(args.config_file)\n    cfg.merge_from_list(args.opts)\n    cfg.freeze()\n    base_train.default_setup(cfg, args)\n    return cfg\n\n\ndef main(args):\n    base_train.setup = setup\n    return base_train.main(args)\n\n\nif __name__ == "__main__":\n    args = base_train.default_argument_parser().parse_args()\n    args.eval_only = False\n    print("Command Line Args:", args)\n    base_train.launch(\n        main,\n        args.num_gpus,\n        num_machines=args.num_machines,\n        machine_rank=args.machine_rank,\n        dist_url=args.dist_url,\n        args=(args,),\n    )\n''')\n\nPath('/content/U2Seg/configs/COCO-PanopticSegmentation/u2seg_swin_tiny_800.yaml').write_text('''_BASE_: "Base-Panoptic-FPN.yaml"\nMODEL:\n  PIXEL_MEAN: [123.675, 116.280, 103.530]\n  PIXEL_STD: [58.395, 57.120, 57.375]\n  WEIGHTS: "/content/U2Seg/weights/swin_tiny_patch4_window7_224.pth"\n  BACKBONE:\n    NAME: "build_swin_fpn_backbone"\n    FREEZE_AT: 0\n  SWIN:\n    PRETRAIN_IMG_SIZE: 224\n    PATCH_SIZE: 4\n    EMBED_DIM: 96\n    DEPTHS: [2, 2, 6, 2]\n    NUM_HEADS: [3, 6, 12, 24]\n    WINDOW_SIZE: 7\n    MLP_RATIO: 4.0\n    QKV_BIAS: True\n    DROP_RATE: 0.0\n    ATTN_DROP_RATE: 0.0\n    DROP_PATH_RATE: 0.2\n    APE: False\n    PATCH_NORM: True\n    OUT_INDICES: [0, 1, 2, 3]\n    USE_CHECKPOINT: False\n  FPN:\n    IN_FEATURES: ["p0", "p1", "p2", "p3"]\n    NORM: "SyncBN"\n  SEM_SEG_HEAD:\n    NUM_CLASSES: 28\n  ROI_BOX_HEAD:\n    CLS_AGNOSTIC_BBOX_REG: True\n  RPN:\n    POST_NMS_TOPK_TRAIN: 4000\n    NMS_THRESH: 0.65\n  ROI_HEADS:\n    NAME: CascadeROIHeads\n    NUM_CLASSES: 800\n\nSOLVER:\n  STEPS: (210000, 250000)\n  MAX_ITER: 270000\n  IMS_PER_BATCH: 16\n  BASE_LR: 0.01\n  WEIGHT_DECAY: 0.00005\n  GAMMA: 0.02\n  CLIP_GRADIENTS:\n    CLIP_TYPE: norm\n    CLIP_VALUE: 1.0\n    ENABLED: true\n    NORM_TYPE: 2.0\n  AMP:\n    ENABLED: True\n  CHECKPOINT_PERIOD: 10000\n\nINPUT:\n  MIN_SIZE_TRAIN: (240, 320, 480, 640, 672, 704, 736, 768, 800, 1024)\n  MAX_SIZE_TRAIN: 1333\n  MASK_FORMAT: "bitmask"\n  FORMAT: "RGB"\nTEST:\n  EVAL_PERIOD: 0\n  PRECISE_BN:\n    ENABLED: True\n    NUM_ITER: 200\n  DETECTIONS_PER_IMAGE: 100\n\nOUTPUT_DIR: ./test_800_cocotrain_u2seg_swin_tiny\n''')\nprint('Injected Swin files.')

In [ ]:
# Step 1 (opsional, butuh panoptic_train2017.json valid).\n# Jalankan jika ingin regenerate pseudo panoptic train.\n# !python datasets/prepare_ours/generate_pseudo_panoptic.py --class_num 800 --split train\n\n# Step 2 (wajib jika folder stuff belum ada).\n!python datasets/prepare_ours/prepare_stuff_panoptic_fpn.py --cluster_num 800 --split train

In [ ]:
# Download Swin pretrained + pastikan instance JSON valid untuk loader COCO.\n!mkdir -p weights\n!test -f weights/swin_tiny_patch4_window7_224.pth || curl -L --fail -o weights/swin_tiny_patch4_window7_224.pth https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth\n\nimport json, copy\nfrom pathlib import Path\n\nroot = Path('datasets/prepare_ours/u2seg_annotations')\nins_dir = root / 'ins_annotations'\npan_dir = root / 'panoptic_annotations'\nsrc_panoptic = ins_dir / 'cocotrain_800_ins_panoptic.json'\ndst_instance = ins_dir / 'cocotrain_800.json'\nstuff_dir = pan_dir / 'panoptic_stuff_cocotrain_800'\ntrain_img_dir = Path('datasets/coco/train2017')\n\nsrc = json.load(open(src_panoptic, 'r'))\nrebuild = True\nif dst_instance.exists():\n    try:\n        cur = json.load(open(dst_instance, 'r'))\n        anns = cur.get('annotations', [])\n        rebuild = not (isinstance(anns, list) and (len(anns) == 0 or ('id' in anns[0] and 'image_id' in anns[0])))\n    except Exception:\n        rebuild = True\n\nif rebuild:\n    out = {\n        'images': copy.deepcopy(src.get('images', [])),\n        'annotations': [],\n        'categories': copy.deepcopy(src.get('categories', [])),\n        'info': copy.deepcopy(src.get('info', {})),\n        'licenses': copy.deepcopy(src.get('licenses', [])),\n    }\n    ann_id = 1\n    for image_id_str, item in src.get('annotations', {}).items():\n        image_id = int(image_id_str)\n        for seg in item.get('segments_info', []):\n            x, y, w, h = seg.get('bbox', [0, 0, 0, 0])\n            out['annotations'].append({\n                'id': ann_id,\n                'image_id': seg.get('image_id', image_id),\n                'category_id': int(seg['category_id']),\n                'bbox': [float(x), float(y), float(w), float(h)],\n                'area': float(seg.get('area', w * h)),\n                'iscrowd': int(seg.get('iscrowd', 0)),\n                'segmentation': seg['segmentation'],\n            })\n            ann_id += 1\n    json.dump(out, open(dst_instance, 'w'), ensure_ascii=False)\n\ndata = json.load(open(dst_instance, 'r'))\nimage_name_set = {p.name for p in train_img_dir.glob('*.jpg')}\nstuff_name_set = {p.name.replace('.png', '.jpg') for p in stuff_dir.glob('*.png')}\nkeep_names = image_name_set & stuff_name_set\nkeep_images = [im for im in data.get('images', []) if im.get('file_name') in keep_names]\nkeep_ids = {im['id'] for im in keep_images}\ndata['images'] = keep_images\ndata['annotations'] = [ann for ann in data.get('annotations', []) if ann.get('image_id') in keep_ids]\njson.dump(data, open(dst_instance, 'w'), ensure_ascii=False)\nprint('Final cocotrain_800.json:', len(data['images']), 'images /', len(data['annotations']), 'annotations')

In [ ]:
# Long training (resume-able), eval sementara dimatikan agar training stabil dulu.\n%cd /content/U2Seg\n!export CLUSTER_NUM=800 && python tools/train_net_swin.py --num-gpus 1 --resume --config-file configs/COCO-PanopticSegmentation/u2seg_swin_tiny_800.yaml TEST.PRECISE_BN.ENABLED False DATASETS.TEST '()'